In [ ]:
# 导包
import time

import matplotlib.pyplot as plt
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchsummary import summary
from torchvision.datasets import CIFAR10
from torchvision.transforms import (
    ToTensor,  # pip install torchvision -i https://mirrors.aliyun.com/pypi/simple/
)

print("CUDA是否可用:", torch.cuda.is_available())
print("PyTorch版本:", torch.__version__)
print("CUDA版本:", torch.version.cuda)

In [ ]:
# 每批次样本数
BATCH_SIZE = 8

In [ ]:
# 1.创建数据集
def create_dataset():
    # 加载训练集
    train_dataset = CIFAR10(root='./data', train=True, transform=ToTensor(), download=True)
    # 加载测试集
    test_dataset = CIFAR10(root='./data', train=False, transform=ToTensor(), download=True)
    # 返回数据集
    return train_dataset, test_dataset

In [ ]:
# 2.搭建神经网络
class ImgModel(nn.Module):

    # 初始化
    def __init__(self):
        super().__init__()
        # 第1个卷积层    输入3通道 输出6通道 卷积核3*3 默认 padding=0 stride=1
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=6, kernel_size=3)
        # 第1个池化层    输入3通道 输出3通道 池化核3*3 默认 padding=0 stride=1
        self.pool1 = nn.MaxPool2d(2, 2)

        # 第2个卷积层    输入6通道 输出16通道 卷积核3*3 默认 padding=0 stride=1
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=3)
        # 第2个池化层    输入3通道 输出3通道 池化核3*3 默认 pa1dding=0 stride=1
        self.pool2 = nn.MaxPool2d(2, 2)

        # 第1个全连接层   输入：576 输出：120
        self.linear1 = nn.Linear(576, 120)
        # 第2个全连接层   输入：120 输出：84
        self.linear2 = nn.Linear(120, 84)
        # 输出层   输入：84 输出：10
        self.output = nn.Linear(84, 10)

    # 前向传播
    def forward(self, x):
        # 第1层：卷积层 + 激励层(激活函数) + 池化层
        x = self.pool1(torch.relu(self.conv1(x)))
        # 第2层：卷积层 + 激励层(激活函数) + 池化层
        x = self.pool2(torch.relu(self.conv2(x)))

        # 将数据拉平(展开成2维)
        x = x.reshape(x.size(0), -1)

        # 第3层：全连接层 + 激励层
        x = torch.relu(self.linear1(x))
        # 第4层：全连接层 + 激励层
        x = torch.relu(self.linear2(x))
        # 输出层
        x = self.output(x)
        # 返回数据
        return x

In [ ]:
# 3.模型训练
def train(model, dataset):
    # 定义设备（自动检测 GPU）
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'当前使用计算设备: {device}')

    # 加载数据集
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    total_samples = len(dataloader.dataset)  # 总样本数
    print(f'当前一轮训练的数据集样本数: {total_samples}')

    # 将模型移至 GPU
    model = model.to(device)

    # 损失函数
    criterion = nn.CrossEntropyLoss()
    # 优化器
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    # 切换模型到训练状态
    model.train()

    # 定义变量
    epochs = 10  # 总训练轮数
    loss_list = []  # 每轮的平均损失
    total_time = 0  # 总耗时
    acc_list = []  # 准确率列表
    # 轮次训练
    for epoch in range(epochs):
        # 定义记录变量
        total_loss = 0  # 该轮的总损失
        total_correct = 0  # 该轮的总正确个数
        start_time = time.time()  # 本轮开始时间
        for x, y in dataloader:
            # 将数据移至 GPU
            x, y = x.to(device), y.to(device)
            # 模型预测
            y_pred = model(x)
            # 损失计算
            loss = criterion(y_pred, y)
            # 梯度清零 + 反向传播 + 参数更新
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # 每批统计
            total_loss += loss.item() * len(y)
            total_correct += torch.sum(torch.argmax(y_pred, dim=1) == y).item()

        # 每轮统计
        loss_list.append(total_loss / total_samples)
        acc_list.append(total_correct / total_samples)
        epoch_time = time.time() - start_time
        total_time += epoch_time
        print(
            f'第{epoch + 1}轮，loss: {loss_list[-1]:.2f} , accuracy: {acc_list[-1] * 100:.2f}% , 该轮耗时: {epoch_time:.2f}s , 总耗时: {total_time:.2f}s')

    # 保存模型
    torch.save(model.cpu().state_dict(), './model/ImgModel.pth')
    model.to(device)  # 放回GPU

    # 可视化训练结果 (Loss 和 Accuracy 建议分开画)
    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.plot(range(1, epochs + 1), loss_list, marker='o')
    plt.title('Training Loss')
    plt.xlabel('Epoch')

    plt.subplot(1, 2, 2)
    plt.plot(range(1, epochs + 1), acc_list, marker='o', color='orange')
    plt.title('Training Accuracy')
    plt.xlabel('Epoch')

    plt.tight_layout()
    plt.show()

In [ ]:
# 测试
train_dataset, test_dataset = create_dataset()
# img, label = train_dataset[1]
# print(img)
# print(f'class: {label}')
# img = torch.permute(img, (1, 2, 0))
# plt.imshow(img)
# plt.show()

model = ImgModel()
# summary(model, input_size=(3, 32, 32))
train(model, train_dataset)
